# Phase 1 · Exercise 2 — What do the signals look like?

**Why this notebook:** see what the experts saw, and check the data formats yourself instead of trusting
community posts. Ex1 only looked at `train.csv`; here we open the raw EEG and the spectrogram files.

**What you will have at the end**

| Output | Used in the writeup for |
|---|---|
| Shape, columns and NaNs of one EEG and one spectrogram file (+ a check on 100 random files) | Methods (data description) |
| Spectrogram time step and frequency grid, measured | Methods; confirms the 2 s / 0.59–19.92 Hz claim |
| Check that EEG and spectrogram offsets are on the same clock | Methods (why the two views can be combined) |
| One figure per class: 16 bipolar "double-banana" channels + the 4 regional spectrograms | Background / Methods figure |
| One sentence per class on what you see | Background (clinical key: L vs G, PD vs RDA) |

All numbers and figures are written to `outputs_02/`.

**Runs on:** Kaggle (attach the competition data) or locally with the data in `data/hms/` at the repo root.
It reads `train.csv` plus ~120 EEG and ~120 spectrogram parquet files, so it takes a minute or two.

## 0. Setup

In [ ]:
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt

# Same data finder as Ex1: on Kaggle search /kaggle/input, locally use data/hms (repo README).
def find_data_dir():
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        for csv in sorted(kaggle.rglob("train.csv")):
            if (csv.parent / "train_spectrograms").exists():
                return csv.parent
    for p in [Path("../data/hms"), Path("data/hms")]:
        if (p / "train.csv").exists():
            return p
    return None


DATA_DIR = find_data_dir()
if DATA_DIR is None:
    attached = sorted(str(p) for p in Path("/kaggle/input").glob("*/*")) if Path("/kaggle/input").exists() else []
    raise FileNotFoundError(
        "Competition data not found. On Kaggle: right panel -> 'Add Input' -> Competitions -> "
        "'HMS - Harmful Brain Activity Classification' -> Add. Locally: put the data in data/hms/.\n"
        f"Currently under /kaggle/input: {attached or 'nothing'}")
print("Data directory:", DATA_DIR.resolve())
EEG_DIR, SPEC_DIR = DATA_DIR / "train_eegs", DATA_DIR / "train_spectrograms"

OUT_DIR = Path("/kaggle/working/outputs_02") if Path("/kaggle/working").exists() else Path("outputs_02")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs go to:  ", OUT_DIR.resolve())

VOTE_COLS = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
CLASS_NAMES = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]
HIGH_VOTES = 10                  # same "reliable label" threshold as Ex1
IDEAL_MIN, EDGE_MAX_GAP = 0.9, 0.2   # same idealized / edge rule as Ex1
SEED = 42

FS = 200                         # EEG sampling rate (Hz) -- checked below
EEG_WIN_S = 50                   # one labelled EEG window = 50 s = 10,000 rows
SPEC_WIN_S = 600                 # one labelled spectrogram view = 10 min
N_PER_CLASS = 2                  # high-agreement examples to plot per class
PLOT_SECONDS = 20                # EEG seconds shown around the centre (the labelled 10 s is shaded)
N_CHECK = 100                    # random files used for the format check over many files

pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Pick high-agreement examples per class

Same soft labels and idealized/proto/edge rule as Ex1. Per class we take `N_PER_CLASS` windows with ≥ 10 votes,
preferring **idealized** ones (top vote share ≥ 0.9), from different patients. The three idealized examples
shown in Ex1 (`outputs_01/agreement_examples.csv`) are reused first so the two notebooks tell one story.
If a class has too few idealized windows, we fall back to its 50 most-agreed windows.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
train["total_votes"] = train[VOTE_COLS].sum(axis=1)
Y = train[VOTE_COLS].to_numpy(float)
Y /= Y.sum(axis=1, keepdims=True)
sorted_p = np.sort(Y, axis=1)[:, ::-1]
train["p1"], train["p2"] = sorted_p[:, 0], sorted_p[:, 1]
train["agreement"] = np.select([train.p1 >= IDEAL_MIN, train.p1 - train.p2 <= EDGE_MAX_GAP],
                               ["idealized", "edge"], "proto")
H = train[train.total_votes >= HIGH_VOTES]

# Idealized examples from Ex1 (label_id), reused here.
EX1_LABEL_IDS = {"GPD": 1236367216, "GRDA": 1309158704, "Seizure": 1587277164}

picks = []
for cls in CLASS_NAMES:
    cand = H[H.expert_consensus == cls].sort_values(["p1", "total_votes"], ascending=False)
    ideal = cand[cand.agreement == "idealized"]
    pool = ideal if len(ideal) >= N_PER_CLASS else cand.head(50)
    pool = pool.drop_duplicates("patient_id").sample(frac=1, random_state=SEED)  # shuffle, 1 window per patient
    first = train[train.label_id == EX1_LABEL_IDS.get(cls, -1)]
    if len(first):
        pool = pd.concat([first, pool[pool.patient_id != first.patient_id.iloc[0]]])
    picks.append(pool.head(N_PER_CLASS).assign(n_idealized_in_class=len(ideal)))

examples = pd.concat(picks).reset_index(drop=True)
EX_COLS = ["expert_consensus", "agreement", "p1", "total_votes", "label_id", "patient_id", "eeg_id",
           "eeg_label_offset_seconds", "spectrogram_id", "spectrogram_label_offset_seconds", "n_idealized_in_class"]
examples[EX_COLS]

## 2. Open one EEG file and one spectrogram file

Expected (from the plan, Q5): EEG = 200 Hz, 19 electrodes + EKG = 20 columns, one file can hold many 50-s windows.
Spectrogram = a `time` column + 400 columns (LL/RL/LP/RP × 100 frequencies). Check it rather than believe it.

In [ ]:
row0 = examples.iloc[0]
eeg = pd.read_parquet(EEG_DIR / f"{row0.eeg_id}.parquet")
print(f"EEG file {row0.eeg_id}: shape {eeg.shape}")
print("Columns:", list(eeg.columns))
print("dtypes: ", eeg.dtypes.value_counts().to_dict())
print(f"Duration at {FS} Hz: {len(eeg) / FS:.1f} s")
print("NaNs per column:", eeg.isna().sum().to_dict())

# Does the file length match the windows train.csv cuts from it?
rec = train[train.eeg_id == row0.eeg_id]
need = int((rec.eeg_label_offset_seconds.max() + EEG_WIN_S) * FS)
print(f"\n{len(rec)} labelled windows in this file; last offset {rec.eeg_label_offset_seconds.max():.0f} s "
      f"-> needs {need} rows, file has {len(eeg)}")
eeg.describe().T.round(1).head(20)

In [ ]:
spec = pd.read_parquet(SPEC_DIR / f"{row0.spectrogram_id}.parquet")
print(f"Spectrogram file {row0.spectrogram_id}: shape {spec.shape}")
print("First columns:", list(spec.columns[:5]), "... last:", list(spec.columns[-3:]))
print("time column, first values:", spec.time.head(5).tolist(), " last:", spec.time.iloc[-1])
print(f"NaN cells: {int(spec.isna().sum().sum())} of {spec.size} ({spec.isna().mean().mean():.2%})")
print("Rows with any NaN:", int(spec.isna().any(axis=1).sum()))
spec.iloc[:5, :8]

## 3. Spectrogram time step and frequency grid

The community says a row every 2 s and 100 frequencies from 0.59 to 19.92 Hz per region. Measure it from the
column names and the `time` column. Also check whether the frequency spacing is uniform.

In [ ]:
freq_cols = [c for c in spec.columns if c != "time"]
REGIONS = list(dict.fromkeys(c.split("_")[0] for c in freq_cols))   # keeps file order
REGION_COLS = {r: [c for c in freq_cols if c.split("_")[0] == r] for r in REGIONS}
freqs = {r: np.array([float(c.split("_")[1]) for c in REGION_COLS[r]]) for r in REGIONS}
FREQS = freqs[REGIONS[0]]
same_grid = all(np.array_equal(freqs[r], FREQS) for r in REGIONS)

dt = np.diff(spec.time.to_numpy())
df_ = np.diff(FREQS)
print("Regions in file order:", REGIONS, "| same frequency grid in every region:", same_grid)
print(f"Frequencies per region: {len(FREQS)}, from {FREQS.min()} to {FREQS.max()} Hz")
print("Frequency steps (rounded):", pd.Series(df_.round(2)).value_counts().to_dict())
print("Time steps in this file:", pd.Series(dt).value_counts().to_dict())
print(f"Rows in a {SPEC_WIN_S}-s view at that step: {SPEC_WIN_S / np.median(dt):.0f}")

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(FREQS, ".", ms=3)
ax.set(xlabel="column index within a region", ylabel="frequency (Hz)", title="Spectrogram frequency grid")
plt.tight_layout()
plt.savefig(OUT_DIR / "spectrogram_frequency_grid.png", bbox_inches="tight")
plt.show()

### 3b. Same checks over many files

One file proves little. Read `N_CHECK` random EEG and spectrogram files and check that every file has the same
columns, the same time step, and how many NaNs there are. EEG length should be at least what `train.csv` needs.

In [ ]:
rng = np.random.default_rng(SEED)
need_rows = (train.groupby("eeg_id").eeg_label_offset_seconds.max() + EEG_WIN_S) * FS

eeg_rows = []
for eid in rng.choice(train.eeg_id.unique(), N_CHECK, replace=False):
    e = pd.read_parquet(EEG_DIR / f"{eid}.parquet")
    eeg_rows.append({"eeg_id": eid, "rows": len(e), "same_columns": list(e.columns) == list(eeg.columns),
                     "nan_frac": float(e.isna().mean().mean()), "rows_needed": int(need_rows[eid])})
eeg_check = pd.DataFrame(eeg_rows)
eeg_check["long_enough"] = eeg_check.rows >= eeg_check.rows_needed

spec_rows = []
for sid in rng.choice(train.spectrogram_id.unique(), N_CHECK, replace=False):
    s = pd.read_parquet(SPEC_DIR / f"{sid}.parquet")
    steps = np.unique(np.diff(s.time.to_numpy()))
    spec_rows.append({"spectrogram_id": sid, "rows": len(s), "same_columns": list(s.columns) == list(spec.columns),
                      "time_steps": ",".join(f"{x:g}" for x in steps), "time_first": s.time.iloc[0],
                      "nan_frac": float(s.isna().mean().mean())})
spec_check = pd.DataFrame(spec_rows)

print(f"EEG  ({N_CHECK} files): same columns {eeg_check.same_columns.mean():.0%}, long enough "
      f"{eeg_check.long_enough.mean():.0%}, files with any NaN {(eeg_check.nan_frac > 0).mean():.0%}, "
      f"duration {eeg_check.rows.min() / FS:.0f}-{eeg_check.rows.max() / FS:.0f} s")
print(f"Spec ({N_CHECK} files): same columns {spec_check.same_columns.mean():.0%}, files with any NaN "
      f"{(spec_check.nan_frac > 0).mean():.0%}, time steps seen {spec_check.time_steps.value_counts().to_dict()}, "
      f"first time value {spec_check.time_first.value_counts().head(3).to_dict()}")
eeg_check.to_csv(OUT_DIR / "file_check_eeg.csv", index=False)
spec_check.to_csv(OUT_DIR / "file_check_spectrogram.csv", index=False)

### 3c. Are the EEG and the spectrogram on the same clock?

If both offsets refer to the same moment, then for one EEG and its spectrogram the difference
`spectrogram_label_offset_seconds − eeg_label_offset_seconds` must be the same for every window
(the EEG file just starts later inside the longer spectrogram recording). Checked on `train.csv` only.

In [ ]:
diff = train.spectrogram_label_offset_seconds - train.eeg_label_offset_seconds
n_diff = diff.groupby([train.eeg_id, train.spectrogram_id]).nunique()
print(f"(eeg_id, spectrogram_id) pairs: {len(n_diff)}; with one constant offset difference: {(n_diff == 1).mean():.2%}")
print("Spectrograms per eeg_id:", train.groupby("eeg_id").spectrogram_id.nunique().value_counts().to_dict())

## 4. Cut the windows for one `label_id`

* EEG: rows `eeg_label_offset_seconds × 200` to `+ 10,000` (50 s). The **labelled 10 s** are the centre, rows 4,000–6,000.
* Spectrogram: rows with `time` in `[offset, offset + 600)`. The 50-s EEG sits in the middle of these 10 minutes.

In [ ]:
@lru_cache(maxsize=64)
def load_eeg(eeg_id):
    return pd.read_parquet(EEG_DIR / f"{eeg_id}.parquet")


@lru_cache(maxsize=64)
def load_spec(spectrogram_id):
    return pd.read_parquet(SPEC_DIR / f"{spectrogram_id}.parquet")


def eeg_window(row):
    start = int(row.eeg_label_offset_seconds * FS)
    win = load_eeg(int(row.eeg_id)).iloc[start:start + EEG_WIN_S * FS].reset_index(drop=True)
    assert len(win) == EEG_WIN_S * FS, f"EEG window too short: {len(win)} rows"
    return win


def spec_window(row):
    s = load_spec(int(row.spectrogram_id))
    off = row.spectrogram_label_offset_seconds
    return s[(s.time >= off) & (s.time < off + SPEC_WIN_S)].reset_index(drop=True)


ew, sw = eeg_window(row0), spec_window(row0)
print(f"label_id {row0.label_id} ({row0.expert_consensus})")
print(f"EEG window:  {ew.shape}, NaN cells {int(ew.isna().sum().sum())}")
print(f"Spec window: {sw.shape}, time {sw.time.iloc[0]:g}..{sw.time.iloc[-1]:g} s, NaN cells {int(sw.isna().sum().sum())}")

# Every example window must cut cleanly.
examples["eeg_nan_frac"] = [eeg_window(r).isna().mean().mean() for r in examples.itertuples()]
examples["spec_rows"] = [len(spec_window(r)) for r in examples.itertuples()]
examples["spec_nan_frac"] = [spec_window(r).isna().mean().mean() for r in examples.itertuples()]
examples[["expert_consensus", "label_id", "eeg_nan_frac", "spec_rows", "spec_nan_frac"]]

## 5. Double-banana bipolar montage

Clinicians do not read the 19 raw electrodes; they read differences between neighbours along four chains
(16 channels). We also band-pass 0.5–20 Hz (the range the patterns live in, and the spectrogram's range), which removes
slow drift and 60 Hz mains noise. NaNs are linearly interpolated for plotting only.

In [ ]:
CHAINS = {"LL": ["Fp1", "F7", "T3", "T5", "O1"],
          "LP": ["Fp1", "F3", "C3", "P3", "O1"],
          "RP": ["Fp2", "F4", "C4", "P4", "O2"],
          "RL": ["Fp2", "F8", "T4", "T6", "O2"]}
CHAIN_COLORS = {"LL": "tab:blue", "LP": "tab:cyan", "RP": "tab:orange", "RL": "tab:red"}
BAND_SOS = butter(4, [0.5, 20], btype="band", fs=FS, output="sos")


def bipolar(win, filt=True):
    x = win.interpolate(limit_direction="both").fillna(0)
    chans, names, chain_of = [], [], []
    for chain, el in CHAINS.items():
        for a, b in zip(el[:-1], el[1:]):
            chans.append(x[a].to_numpy() - x[b].to_numpy())
            names.append(f"{a}-{b}")
            chain_of.append(chain)
    X = np.stack(chans, axis=1)
    if filt:
        X = sosfiltfilt(BAND_SOS, X, axis=0)
    return X, names, chain_of


def plot_eeg(ax, row, seconds=PLOT_SECONDS):
    X, names, chain_of = bipolar(eeg_window(row))
    c, half = EEG_WIN_S * FS // 2, int(seconds * FS / 2)
    X = X[c - half:c + half]
    t = np.arange(-half, half) / FS
    spacing = 2 * np.median(np.percentile(np.abs(X), 95, axis=0)) or 1.0   # robust per-figure scale
    pos, ticks = 0.0, []
    for i, (name, chain) in enumerate(zip(names, chain_of)):
        if i and chain != chain_of[i - 1]:
            pos += 0.6                                                     # small gap between chains
        ax.plot(t, X[:, i] - pos * spacing, lw=0.5, color=CHAIN_COLORS[chain])
        ticks.append(-pos * spacing)
        pos += 1
    ax.set_yticks(ticks)
    ax.set_yticklabels(names, fontsize=6)
    ax.axvspan(-5, 5, color="gold", alpha=0.15, lw=0)
    ax.set_xlim(t[0], t[-1])
    ax.set_xlabel("time from window centre (s)", fontsize=8)
    ax.tick_params(axis="x", labelsize=7)
    return spacing


fig, ax = plt.subplots(figsize=(10, 5))
sp = plot_eeg(ax, row0)
ax.set_title(f"{row0.expert_consensus}, label_id {row0.label_id}: bipolar montage, 0.5-20 Hz "
             f"(trace spacing {sp:.0f} µV, shaded = labelled 10 s)", fontsize=9)
plt.tight_layout()
plt.show()

## 6. Spectrograms

Power is log10-transformed (it spans orders of magnitude), with one colour scale shared by the 4 regions of a window
so left and right can be compared. Time is relative to the label centre; dashed lines mark the 50-s EEG window.

In [ ]:
LOG_FLOOR = 1e-2


def plot_spec(axes, row):
    w = spec_window(row)
    t = w.time.to_numpy() - (row.spectrogram_label_offset_seconds + SPEC_WIN_S / 2)
    logp = {r: np.log10(np.clip(w[REGION_COLS[r]].to_numpy(float), LOG_FLOOR, None)) for r in REGIONS}
    allv = np.concatenate([v.ravel() for v in logp.values()])
    vmin, vmax = np.nanpercentile(allv, [2, 98]) if np.isfinite(allv).any() else (0, 1)
    for ax, r in zip(axes, ["LL", "LP", "RP", "RL"]):
        im = ax.imshow(logp[r].T, aspect="auto", origin="lower", cmap="viridis", vmin=vmin, vmax=vmax,
                       extent=[t[0], t[-1], FREQS[0], FREQS[-1]], interpolation="nearest")
        for x in (-EEG_WIN_S / 2, EEG_WIN_S / 2):
            ax.axvline(x, color="white", ls="--", lw=0.7)
        ax.set_ylabel(f"{r}\nHz", fontsize=7, color=CHAIN_COLORS[r])
        ax.tick_params(labelsize=6)
    axes[-1].set_xlabel("time from label centre (s)", fontsize=8)
    return im


fig, axes = plt.subplots(4, 1, figsize=(8, 6), sharex=True)
im = plot_spec(axes, row0)
fig.colorbar(im, ax=axes, label="log10 power", shrink=0.8)
axes[0].set_title(f"{row0.expert_consensus}, label_id {row0.label_id}: 10-min spectrogram", fontsize=9)
plt.show()

## 7. One figure per example: EEG + spectrograms

Left: 16 bipolar channels around the labelled 10 s (blue = left chains, red/orange = right chains).
Right: the 4 regional spectrograms (same order top-to-bottom: LL, LP, RP, RL). Each figure is saved to `outputs_02/`.

In [ ]:
def plot_example(fig, row):
    gs = fig.add_gridspec(4, 2, width_ratios=[2.2, 1], hspace=0.15, wspace=0.12)
    ax_eeg = fig.add_subplot(gs[:, 0])
    sp = plot_eeg(ax_eeg, row)
    spec_axes = [fig.add_subplot(gs[i, 1]) for i in range(4)]
    plot_spec(spec_axes, row)
    for ax in spec_axes[:-1]:
        ax.tick_params(labelbottom=False)
    votes = ", ".join(f"{n} {int(getattr(row, v))}" for n, v in zip(CLASS_NAMES, VOTE_COLS) if getattr(row, v))
    ax_eeg.set_title(f"{row.expert_consensus} ({row.agreement}, p1 = {row.p1:.2f}) | label_id {row.label_id}, "
                     f"patient {row.patient_id} | votes: {votes} | spacing {sp:.0f} µV", fontsize=8, loc="left")


for row in examples.itertuples():
    fig = plt.figure(figsize=(14, 5))
    plot_example(fig, row)
    fig.savefig(OUT_DIR / f"example_{row.expert_consensus}_{row.label_id}.png", bbox_inches="tight")
    plt.show()

### Writeup figure: one example per class

The first example of each class in one figure (`outputs_02/examples_per_class.png`).

In [ ]:
first_per_class = examples.groupby("expert_consensus", sort=False).head(1)
fig = plt.figure(figsize=(14, 4.2 * len(first_per_class)))
for sub, row in zip(fig.subfigures(len(first_per_class), 1), first_per_class.itertuples()):
    plot_example(sub, row)
fig.savefig(OUT_DIR / "examples_per_class.png", bbox_inches="tight", dpi=150)
plt.show()

## 8. A first number for "left vs right"

The clinical key says L-classes (LPD, LRDA) are one-sided and G-classes (GPD, GRDA) are on both sides. A crude check
(and a preview of an Ex3 feature): mean log-power of the left chains minus the right chains, 0.5–20 Hz,
over the central 50 s of the spectrogram. Expect large |asymmetry| for LPD/LRDA and values near 0 for GPD/GRDA.
With only a few examples per class this is an illustration, not evidence.

In [ ]:
def lr_asymmetry(row):
    w = spec_window(row)
    t = w.time.to_numpy() - (row.spectrogram_label_offset_seconds + SPEC_WIN_S / 2)
    c = np.abs(t) <= EEG_WIN_S / 2
    m = {r: np.nanmean(np.log10(np.clip(w.loc[c, REGION_COLS[r]].to_numpy(float), LOG_FLOOR, None)))
         for r in REGIONS}
    return (m["LL"] + m["LP"]) / 2 - (m["RL"] + m["RP"]) / 2


examples["lr_asym"] = [lr_asymmetry(r) for r in examples.itertuples()]
examples[["expert_consensus", "label_id", "lr_asym"]].round(3)

## 9. What I see, per class

Look at the per-class figures and write **one sentence per class**. The "what to look for" column is the ACNS
(Hirsch 2021) description in plain words; the last column is yours.

| Class | What to look for | What I see (fill in) |
|---|---|---|
| Seizure | Rhythmic activity that **evolves** (frequency / amplitude / location change); on the spectrogram a burst of power across frequencies ("flame") | … |
| LPD | Sharp **blips** repeating at regular intervals with flat gaps, mainly on **one side** (blue or red chains) | … |
| GPD | Same periodic blips, but on **both sides**, roughly symmetric | … |
| LRDA | **Continuous** slow (delta, ~1–4 Hz) rhythmic waves, no gaps, on **one side** | … |
| GRDA | Continuous rhythmic delta on **both sides**, often strongest frontally (Fp/F channels) | … |
| Other | None of the above: background slowing, artifact, or normal activity | … |

## 10. Save outputs

In [ ]:
checks = pd.Series({
    "eeg_columns": len(eeg.columns),
    "eeg_column_names": " ".join(eeg.columns),
    "eeg_window_rows": EEG_WIN_S * FS,
    "spec_columns_incl_time": spec.shape[1],
    "spec_regions": " ".join(REGIONS),
    "spec_freqs_per_region": len(FREQS),
    "spec_freq_min_hz": FREQS.min(),
    "spec_freq_max_hz": FREQS.max(),
    "spec_freq_steps_hz": " ".join(f"{x:g}" for x in np.unique(df_.round(2))),
    "spec_time_step_s": " ".join(f"{x:g}" for x in np.unique(dt)),
    "spec_window_rows_example": len(sw),
    "eeg_files_checked_same_columns": eeg_check.same_columns.mean(),
    "eeg_files_checked_long_enough": eeg_check.long_enough.mean(),
    "eeg_files_checked_with_nan": (eeg_check.nan_frac > 0).mean(),
    "spec_files_checked_same_columns": spec_check.same_columns.mean(),
    "spec_files_checked_with_nan": (spec_check.nan_frac > 0).mean(),
    "eeg_spec_pairs_constant_offset_diff": (n_diff == 1).mean(),
})
checks.to_csv(OUT_DIR / "format_checks.csv", header=["value"])
examples[EX_COLS + ["eeg_nan_frac", "spec_rows", "spec_nan_frac", "lr_asym"]].to_csv(
    OUT_DIR / "examples_02.csv", index=False)
print("Saved:", sorted(p.name for p in OUT_DIR.iterdir()))
checks

## 11. Summary for the writeup

Fill in after running (copy from the outputs above):

| Item | Value |
|---|---|
| EEG file format (columns, rate, window rows) | … |
| Spectrogram format (columns, regions × freqs) | … |
| Spectrogram time step / frequency range and spacing | … |
| Files with NaNs (EEG / spectrogram, of 100 checked) | … |
| EEG and spectrogram offsets on the same clock? | … |

**Writeup hooks**
- *Methods (data):* one sentence each on the EEG and spectrogram formats, with the measured time step and frequency grid
  (replaces the community-reported numbers), and how NaNs will be handled in Ex3.
- *Background / Methods figure:* `examples_per_class.png`, caption with the one-sentence observations from section 9.
- *Feeds Ex3:* NaN share → cleaning step; log-power scale → log-transform; left-right asymmetry → L vs G feature.